In [3]:
# -*- coding: utf-8 -*-
# 【CP3-02 持久化检查点】PostgresSaver 落盘，跨进程/重启仍可恢复
# 文件：CP3/02_in_SQL.ipynb
# 作用：本 Cell 演示 【CP3-02 持久化检查点】PostgresSaver 落盘，跨进程/重启仍可恢复 的完整可运行示例
# 阅读顺序：状态定义 → 节点定义 → 图构建 → 编译执行 → 结果观察

import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.postgres import PostgresSaver  # 持久化检查点：落 Postgres，跨重启可恢复
from langgraph.graph import MessagesState, StateGraph,START,END

load_dotenv(override=True)
DB_URL = os.getenv('LANGGRAPH_DB_URL')
if not DB_URL:
    raise RuntimeError('缺少 LANGGRAPH_DB_URL，请配置 PostgreSQL 连接串。')

MODEL_NAME = os.getenv('DEEPSEEK_MODEL', 'deepseek-v4-flash')
model = ChatDeepSeek(
    model = MODEL_NAME,
    extra_body=
    {
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 声明状态
class OverAllState(MessagesState):
    output:str

#2. 声明节点
def llm_mode(state:OverAllState) -> OverAllState:
    messages = state["messages"]
    res = model.invoke(messages)
    return {
        "messages":[res]
    }

def output_node(state:OverAllState) -> OverAllState:
    return {
        "output":state["messages"][-1].content
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)  # 创建状态图构建器：绑定状态 Schema
builder.add_node("llm_node",llm_mode)  # 注册节点到图中
builder.add_node("output_node",output_node)  # 注册节点到图中
builder.add_edge(START,"llm_node")  # 起点扇出
builder.add_edge("llm_node","output_node")
builder.add_edge("output_node",END)  # 汇入终点

#4. 配置检查点存储器

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:  # 持久化检查点：落 Postgres，跨重启可恢复
    #5. 第一次使用PostgresSaver作为检查点 需要调用方法 setup()
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)  # 实例化检查点保存器

    # 指定线程ID
    config = {
        "configurable":{
            "thread_id":"chapter03-02"  # 会话标识：同 thread_id 的调用共享同一条记忆链
        }
    }
    # 调用图
    graph.invoke({"messages":[HumanMessage("你好,我是老王")]},config=config)  # 触发图执行：传入初始 State + config
    res = graph.invoke({"messages":[HumanMessage("你好,我是谁")]},config=config)  # 触发图执行：传入初始 State + config
    print(res)


{'messages': [HumanMessage(content='你好,我是谁', additional_kwargs={}, response_metadata={}, id='b6fd36b0-c72a-4bb0-89b0-c915af75e560'), AIMessage(content='你好！😊 从我们的对话来看，我目前还不知道你的具体身份信息呢。你可以告诉我你的名字、或者是想聊些什么，这样我就能更好地认识你，为你提供更贴心的帮助啦！\n\n无论是想聊聊天气、分享心情、探讨问题，还是需要学习、工作上的建议，我都在这儿等着你～✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 8, 'total_tokens': 77, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '08326a75-dfae-4576-a67f-f85db0ba9aad', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a02fed-0a00-7fe3-a694-f33219bdd6c6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 69, 'total_tokens': 77, 'input_token_details': {'cache_read': 0}